# 03 — Feature Engineering

Compute Morgan (ECFP4) fingerprints, apply Lipinski's Rule of Five, and visualize
the chemical space with UMAP.

**Input:** `../data/processed/combined_bioactivity.csv`
**Output:** `features_X.npy`, `features_y.npy`, `features_meta.csv`

In [ ]:
import sys, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from src.features import lipinski_filter, compute_fingerprints, compute_rdkit_descriptors

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

df = pd.read_csv('../data/processed/combined_bioactivity.csv')
print(f"Loaded {len(df)} labeled compounds")

## Lipinski's Rule of Five

Filters out compounds unlikely to be orally bioavailable leads.
Rule: MW ≤ 500, LogP ≤ 5, H-bond donors ≤ 5, H-bond acceptors ≤ 10.

In [ ]:
df_ro5 = lipinski_filter(df)
print(f"After Ro5 filter: {len(df_ro5)} / {len(df)} compounds retained ({len(df_ro5)/len(df):.1%})")

## Compute Morgan fingerprints (ECFP4)

Radius = 2, 2048 bits. Each set bit encodes a circular substructure
centred on an atom out to radius 2 bonds. These are the features the
ML classifier will train on.

In [ ]:
fp_cfg = config['features']
X, valid_df = compute_fingerprints(
    df_ro5,
    radius=fp_cfg['morgan_radius'],
    n_bits=fp_cfg['morgan_n_bits'],
)
y = valid_df['active'].values

print(f"Feature matrix: {X.shape}  |  Active: {y.sum()}  |  Inactive: {(y==0).sum()}")

np.save('../data/processed/features_X.npy', X)
np.save('../data/processed/features_y.npy', y)
valid_df.to_csv('../data/processed/features_meta.csv', index=False)
print("Saved features_X.npy, features_y.npy, features_meta.csv")

## UMAP: visualise chemical space

Project 2048-D fingerprints to 2D. Well-separated active/inactive clusters
suggest the fingerprints carry discriminative signal.

In [ ]:
from umap import UMAP

reducer = UMAP(n_components=2, random_state=42, n_jobs=1)
embedding = reducer.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(embedding[:, 0], embedding[:, 1], c=y, cmap='RdYlGn', alpha=0.4, s=8)
plt.colorbar(sc, label='Active (1) / Inactive (0)')
ax.set_title('Chemical space: UMAP of ECFP4 fingerprints')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
plt.tight_layout()
plt.savefig('../data/processed/umap_chemical_space.png', dpi=150)
plt.show()

## Physicochemical descriptor distributions

In [ ]:
descs = compute_rdkit_descriptors(valid_df)
descs['active'] = y

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flatten(), ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']):
    for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        subset = descs[descs['active'] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, color=color, label='Active' if label else 'Inactive')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Physicochemical properties: active vs inactive', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/descriptor_distributions.png', dpi=150)
plt.show()